In [1]:
import re
import random

def extract_core(name: str) -> str:
    """
    ดึง core action จากชื่อไฟล์
    เช่น:
    01_02__exit_phone_room__YVGY8LOK.mp4
    → exit_phone_room
    """
    name = name.replace(".mp4", "")
    parts = name.split("__")
    if len(parts) >= 2:
        return parts[1]
    return None


In [2]:
from pathlib import Path
from collections import defaultdict

def index_by_core(folder: Path):
    index = defaultdict(list)
    for v in folder.glob("*.mp4"):
        core = extract_core(v.name)
        if core:
            index[core].append(v)
    return index


In [3]:
fake_dir = Path("NEW DATA/FAKE")
real_dir = Path("NEW DATA/REAL")

fake_index = index_by_core(fake_dir)
real_index = index_by_core(real_dir)

common_cores = sorted(set(fake_index) & set(real_index))

print("Matched scenarios:", len(common_cores))
for c in common_cores[:10]:
    print("-", c)


Matched scenarios: 16
- exit_phone_room
- hugging_happy
- kitchen_pan
- kitchen_still
- meeting_serious
- outside_talking_pan_laughing
- outside_talking_still_laughing
- podium_speech_happy
- secret_conversation
- talking_against_wall


In [4]:
def get_matched_videos(fake_index, real_index):
    pairs = []
    for core in fake_index:
        if core in real_index:
            for f in fake_index[core]:
                for r in real_index[core]:
                    pairs.append((core, f, r))
    return pairs

def get_matched_pairs_one_to_one(fake_index, real_index, seed=42):
    """
    คืนค่า list ของ (core, fake_video_path, real_video_path)
    โดย core ต้องมีทั้ง fake และ real
    และเลือกมาอย่างละ 1 (สุ่มแบบ fix seed)
    """
    random.seed(seed)
    pairs = []

    common_cores = sorted(set(fake_index) & set(real_index))

    for core in common_cores:
        fake_v = random.choice(fake_index[core])
        real_v = random.choice(real_index[core])
        pairs.append((core, fake_v, real_v))

    return pairs


pairs = get_matched_pairs_one_to_one(fake_index, real_index)
print("Total matched pairs:", len(pairs))

for p in pairs:
    print(p[0], "→")
    print("  FAKE:", p[1].name)
    print("  REAL:", p[2].name)



Total matched pairs: 16
exit_phone_room →
  FAKE: 18_06__exit_phone_room__H9445M6R.mp4
  REAL: 22__exit_phone_room.mp4
hugging_happy →
  FAKE: 13_11__hugging_happy__61T622EK.mp4
  REAL: 20__hugging_happy.mp4
kitchen_pan →
  FAKE: 18_27__kitchen_pan__BSOLX3SI.mp4
  REAL: 06__kitchen_pan.mp4
kitchen_still →
  FAKE: 18_26__kitchen_still__C9VUQS9O.mp4
  REAL: 18__kitchen_still.mp4
meeting_serious →
  FAKE: 27_26__meeting_serious__IE9N0ZI9.mp4
  REAL: 01__meeting_serious.mp4
outside_talking_pan_laughing →
  FAKE: 07_03__outside_talking_pan_laughing__IFSURI9X.mp4
  REAL: 12__outside_talking_pan_laughing.mp4
outside_talking_still_laughing →
  FAKE: 06_18__outside_talking_still_laughing__M36D0OJT.mp4
  REAL: 19__outside_talking_still_laughing.mp4
podium_speech_happy →
  FAKE: 03_14__podium_speech_happy__7JPPCV50.mp4
  REAL: 06__podium_speech_happy.mp4
secret_conversation →
  FAKE: 20_12__secret_conversation__B0X1CGG2.mp4
  REAL: 18__secret_conversation.mp4
talking_against_wall →
  FAKE: 22_10_

In [ ]:
import cv2
from pathlib import Path

def get_duration_seconds(cap: cv2.VideoCapture) -> float:
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    if fps and fps > 0 and frame_count and frame_count > 0:
        return frame_count / fps
    # fallback: ถ้าหาค่า fps/frame_count ไม่ได้
    ms = cap.get(cv2.CAP_PROP_POS_MSEC)
    return ms / 1000.0 if ms and ms > 0 else 0.0

def read_frame_at_second(video_path: Path, sec: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return None

    # ใช้ timestamp เป็น ms ตรง ๆ (ไม่ต้องคิด fps)
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)

    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    return frame

def extract_paired_frames_by_second(pairs, out_root: str, start_sec: int = 1, step_sec: int = 1):
    out_root = Path(out_root)
    (out_root / "FAKE").mkdir(parents=True, exist_ok=True)
    (out_root / "REAL").mkdir(parents=True, exist_ok=True)

    for core, fake_path, real_path in pairs:
        fake_path = Path(fake_path)
        real_path = Path(real_path)

        cap_f = cv2.VideoCapture(str(fake_path))
        cap_r = cv2.VideoCapture(str(real_path))
        if not cap_f.isOpened() or not cap_r.isOpened():
            print(f"[SKIP] cannot open: {core}")
            cap_f.release(); cap_r.release()
            continue

        dur_f = get_duration_seconds(cap_f)
        dur_r = get_duration_seconds(cap_r)
        cap_f.release(); cap_r.release()

        # จำนวน “วินาทีเต็ม” ที่ทั้งคู่มีร่วมกัน (floor)
        max_sec = int(min(dur_f, dur_r))

        if max_sec < start_sec:
            print(f"[SKIP] too short: {core} (max_sec={max_sec})")
            continue

        saved = 0
        for sec in range(start_sec, max_sec + 1, step_sec):
            f_frame = read_frame_at_second(fake_path, sec)
            r_frame = read_frame_at_second(real_path, sec)

            if f_frame is None or r_frame is None:
                continue

            # ชื่อไฟล์: core__t0001__fake.jpg / core__t0001__real.jpg
            f_out = out_root / "FAKE" / f"{core}__t{sec:04d}__fake.jpg"
            r_out = out_root / "REAL" / f"{core}__t{sec:04d}__real.jpg"

            cv2.imwrite(str(f_out), f_frame)
            cv2.imwrite(str(r_out), r_frame)
            saved += 1

        print(f"[OK] {core}: saved {saved} paired seconds (1..{max_sec})")




In [6]:
# ===== ใช้งาน =====
pairs = get_matched_pairs_one_to_one(fake_index, real_index, seed=42)
extract_paired_frames_by_second(pairs, out_root="Export/paired_frames", start_sec=1, step_sec=1)

[OK] exit_phone_room: saved 16 paired seconds (1..16)
[OK] hugging_happy: saved 32 paired seconds (1..32)
[OK] kitchen_pan: saved 30 paired seconds (1..30)
[OK] kitchen_still: saved 35 paired seconds (1..35)
[OK] meeting_serious: saved 37 paired seconds (1..37)
[OK] outside_talking_pan_laughing: saved 29 paired seconds (1..29)
[OK] outside_talking_still_laughing: saved 27 paired seconds (1..27)
[OK] podium_speech_happy: saved 34 paired seconds (1..34)
[OK] secret_conversation: saved 1 paired seconds (1..1)
[OK] talking_against_wall: saved 37 paired seconds (1..37)
[OK] talking_angry_couch: saved 63 paired seconds (1..63)
[OK] walk_down_hall_angry: saved 17 paired seconds (1..17)
[OK] walking_and_outside_surprised: saved 47 paired seconds (1..47)
[OK] walking_down_indoor_hall_disgust: saved 34 paired seconds (1..34)
[OK] walking_down_street_outside_angry: saved 10 paired seconds (1..10)
[OK] walking_outside_cafe_disgusted: saved 15 paired seconds (1..16)
